## Feature Creation K-12
This script produces the `master_df.csv` which contains K-12 preprocessed CRDC and EdFact Data

In [19]:
# Path to the CRDC data files
CRDC_SCH_DATA_PATH = "./datasets/2017-18-crdc-data-corrected-publication 2/2017-18 Public-Use Files/Data/SCH/CRDC/CSV"
CRDC_FILE_STRUCTURE_PATH = "./datasets/2017-18-crdc-data-corrected-publication 2/2017-18 Public-Use Files/Documentation/2017-18 CRDC File Structure.xlsx"

EDFACTS_SCH_DATA_PATH = "./datasets/2017-18-crdc-data-corrected-publication 2/2017-18 Public-Use Files/Data/SCH/EDFacts/CSV"
EDFACTS_FILE_STRUCTURE_PATH = "./datasets/2017-18-crdc-data-corrected-publication 2/2017-18 Public-Use Files/Documentation/2017-18 EDFacts File Structure.xlsx"

In [20]:
import pandas as pd
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
import os

In [21]:
# Load the column mapping xlsx for column information
crdc_file_structure = pd.read_excel(CRDC_FILE_STRUCTURE_PATH, sheet_name=None)
edfacts_file_structure = pd.read_excel(EDFACTS_FILE_STRUCTURE_PATH, sheet_name=None)

In [22]:
# Reusable function for merging with COMBO_KEY
def merge_with_combo_key(master_df, other_df_path, important_cols):
    dtype_spec = {
        'LEA_ID': 'str',
        'COMBOKEY': 'str',
    }
    other_df = pd.read_csv(other_df_path, encoding='windows-1252', dtype=dtype_spec, usecols=important_cols + ['COMBOKEY'])
    return master_df.merge(other_df, on='COMBOKEY', how='inner')

## The Base Dataframe

The base dataframe for this education data will include the following columns:
- `LEA_STATE` - District State Abbreviation
- `LEA_STATE_NAME` - District State Name
- `LEAID` - 7 Digit LEAID District Identification Code
- `LEA_NAME` - District Name
- `SCHID` - 5 Digit School Identification Code
- `SCH_NAME` - School Name
- `COMBOKEY` - 7 Digit LEAID District Identification Code+5 Digit School Identification Code

Only schools with Elementary, Middle, or High school students are included. Juvenile Justice Facilities and other school types are excluded.

In [23]:
# This df will hold the merged data
master_df = None

path = os.path.join(CRDC_SCH_DATA_PATH, 'School Characteristics.csv')
schools = pd.read_csv(path, encoding='windows-1252', dtype={'COMBOKEY': str, 'SCHID': str, 'LEAID': str})

# Filter out the JJ schools
schools = schools[schools['JJ'] == 'No']

# Filter out schools that don't have any grades
included_grades = ['SCH_GRADE_PS', 'SCH_GRADE_KG', 'SCH_GRADE_G01', 'SCH_GRADE_G02', 'SCH_GRADE_G03', 'SCH_GRADE_G04', 'SCH_GRADE_G05', 'SCH_GRADE_G06', 'SCH_GRADE_G07', 'SCH_GRADE_G08', 'SCH_GRADE_G09', 'SCH_GRADE_G10', 'SCH_GRADE_G11', 'SCH_GRADE_G12']
schools = schools[schools[included_grades].eq('Yes').any(axis=1)]

# Filter out other types of schools
other_schools = ['SCH_STATUS_MAGNET', 'SCH_STATUS_CHARTER', 'SCH_STATUS_ALT', 'SCH_STATUS_SPED']
schools = schools[schools[other_schools].eq('No').all(axis=1)]

base_columns = ["LEA_STATE", "LEA_STATE_NAME", "LEAID", "LEA_NAME", "SCHID", "SCH_NAME", "COMBOKEY"]
master_df = schools[base_columns].copy()
print(master_df.info())
# print(master_df[master_df['LEA_STATE_NAME'] == "VERMONT"])

<class 'pandas.core.frame.DataFrame'>
Index: 80975 entries, 2 to 97631
Data columns (total 7 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   LEA_STATE       80975 non-null  object
 1   LEA_STATE_NAME  80975 non-null  object
 2   LEAID           80975 non-null  object
 3   LEA_NAME        80975 non-null  object
 4   SCHID           80975 non-null  object
 5   SCH_NAME        80975 non-null  object
 6   COMBOKEY        80975 non-null  object
dtypes: object(7)
memory usage: 4.9+ MB
None


### Student Enrollment
**Important Columns:**
- `TOT_ENR_M` -> `total_enrollment_male` - (Overall Student Enrollment: Calculated Male Total)
- `TOT_ENR_F` -> `total_enrollment_female` - (Overall Student Enrollment: Calculated Female Total)

**Exclusions:**
- Schools with enrollment less than or equal to 0

In [24]:
path = os.path.join(CRDC_SCH_DATA_PATH, 'Enrollment.csv')
IMPORTANT_COLUMNS = ['TOT_ENR_M', 'TOT_ENR_F']

# Merge important columns into the master df
master_df = merge_with_combo_key(master_df, path, IMPORTANT_COLUMNS)

master_df = master_df.rename(columns={'TOT_ENR_M': 'total_enrollment_male'})
master_df = master_df.rename(columns={'TOT_ENR_F': 'total_enrollment_female'})

# Filter out schools that don't have any students or have missing data
master_df = master_df[master_df['total_enrollment_male'] > 0]
master_df = master_df[master_df['total_enrollment_female'] > 0]

### School Finances
**Important Columns:**
- `SCH_SAL_TEACH_WFED` -> `teacher_salary_expend_fed` - Federal Salary Expenditures for Teachers
- `SCH_SAL_TEACH_WOFED` -> `teacher_salary_expend` - Salary Expenditures for Teachers Funded with State and Local Funds (no Federal)

**Exclusions:**
- Schools with federal funding less than 0
- Schools with salary expenditures less than or equal to 0
- Schools Districts with multiple schools that enter the same number for teacher expenditures for every school

In [25]:
path = os.path.join(CRDC_SCH_DATA_PATH, 'School Expenditures.csv')
expenditures = pd.read_csv(path, encoding='windows-1252')

expenditures.head()

# Merge important columns into the master df
IMPORTANT_COLUMNS = ['SCH_SAL_TEACH_WFED', 'SCH_SAL_TEACH_WOFED']
master_df = merge_with_combo_key(master_df, path, IMPORTANT_COLUMNS)
master_df = master_df.rename(columns={'SCH_SAL_TEACH_WFED': 'teacher_salary_expend_fed'})
master_df = master_df.rename(columns={'SCH_SAL_TEACH_WOFED': 'teacher_salary_expend'})

# Make teacher salary expend fed only the difference between the two columns
master_df['teacher_salary_expend_fed'] = master_df['teacher_salary_expend_fed'] - master_df['teacher_salary_expend']

# filter out schools where fed funding is less than 0
master_df = master_df[master_df['teacher_salary_expend_fed'] >= 0]

# filter out schools with no teacher/admin salary data or missing data
master_df = master_df[master_df['teacher_salary_expend'] > 0]


# filter out schools that have the same teacher expend for the entire district (data entry error)
master_df = master_df[master_df.groupby('LEAID')['teacher_salary_expend'].transform('std') > 0]

### Staff FTE/Absences
**Important Columns:**
- `SCH_FTETEACH_TOT` -> `teacher_fte` - Total Full-Time Equivalency (FTE) value
- `SCH_FTETEACH_ABSENT` -> `num_teacher_absences` - Number of FTE teachers who were absent more than 10 school days during the school year
- `SCH_FTECOUNSELORS` -> `num_counselors` - Number of FTE school counselors
- `SCH_TEACHERS_CURR_TOT` -> `num_curr_year_teachers` - Number of current school year teachers
- `SCH_TEACHERS_PREV_TOT` -> `num_prev_year_teachers` - Number of previous school year teachers

**Exclusions:**
- Schools with Teacher FTE less than 10

In [26]:
path = os.path.join(CRDC_SCH_DATA_PATH, 'School Support.csv')
expenditures = pd.read_csv(path, encoding='windows-1252')

#print(enrollment.columns.values)
expenditures.head()

# Merge important columns into the master df
IMPORTANT_COLUMNS = ['SCH_FTETEACH_TOT', 'SCH_FTECOUNSELORS', 'SCH_FTETEACH_ABSENT', 'SCH_TEACHERS_CURR_TOT', 'SCH_TEACHERS_PREV_TOT']
master_df = merge_with_combo_key(master_df, path, IMPORTANT_COLUMNS)
master_df = master_df.rename(columns={'SCH_FTETEACH_TOT': 'teachers_fte'})
master_df = master_df.rename(columns={'SCH_FTETEACH_ABSENT': 'num_teacher_absences'})
master_df = master_df.rename(columns={'SCH_FTECOUNSELORS': 'num_counselors'})
master_df = master_df.rename(columns={'SCH_TEACHERS_CURR_TOT': 'num_curr_year_teachers'})
master_df = master_df.rename(columns={'SCH_TEACHERS_PREV_TOT': 'num_prev_year_teachers'})

# filter out schools below teacher fte threshold
master_df = master_df[master_df['teachers_fte'] > 10]

### Custom Features
**Important Columns:**
- `student_teacher_ratio` - The calculated ratio of students per FTE teacher
- `average_teacher_salary` - The average salary per teacher using FTE teacher amount and total teacher salary expend excluding federal funds.
- `average_teacher_salary_adjusted` - The average salary of FTE teachers adjusted to 2017 COL index

**Exclusions:**
- n/a

In [27]:
# Cost of living (provided by the Missouri Economic Research and Information Center)
# https://meric.mo.gov/data/cost-living-data-series
COLI_TO_LEA_STATE_2017 = {
    "AL": 90.3,
    "AK": 131.3,
    "AZ": 95.6,
    "AR": 87.8,
    "CA": 141.0,
    "CO": 102.3,
    "CT": 125.7,
    "DC": 155.7,
    "DE": 102.9,
    "FL": 99.3,
    "GA": 90.8,
    "HI": 188.3,
    "ID": 92.2,
    "IL": 97.2,
    "IN": 91.1,
    "IA": 91.3,
    "KS": 90.2,
    "KY": 93.7,
    "LA": 94.4,
    "ME": 113.6,
    "MD": 128.7,
    "MA": 132.9,
    "MI": 89.7,
    "MN": 99.7,
    "MS": 85.1,
    "MO": 89.9,
    "MT": 100.4,
    "NE": 92.9,
    "NV": 104.7,
    "NH": 115.0,
    "NJ": 121.9,
    "NM": 94.9,
    "NY": 132.5,
    "NC": 94.6,
    "ND": 99.7,
    "OH": 92.3,
    "OK": 89.2,
    "OR": 129.3,
    "PA": 102.0,
    "RI": 123.6,
    "SC": 99.5,
    "SD": 99.5,
    "TN": 89.8,
    "TX": 91.2,
    "UT": 95.7,
    "VT": 120.7,
    "VA": 102.2,
    "WA": 107.1,
    "WV": 95.9,
    "WI": 96.2,
    "WY": 95.6,
    "PR": 104.7
}

# Add new column for student to teacher ratio
master_df['student_teacher_ratio'] = (master_df['total_enrollment_male'] + master_df['total_enrollment_female']) / master_df['teachers_fte']
master_df['student_teacher_ratio'] = master_df['student_teacher_ratio'].round(2)

# Add new column for average teacher salary
master_df['average_teacher_salary'] = master_df['teacher_salary_expend'] / master_df['teachers_fte']
# filter out rows with average teacher salary that is unreasonably high
#master_df = master_df[master_df['average_teacher_salary'] < 200_000]

# Map the COLI values to the LEA_STATE column
master_df['COLI'] = master_df['LEA_STATE'].map(COLI_TO_LEA_STATE_2017)

# Add new column for average teacher salary adjusted for cost of living
master_df['average_teacher_salary_adjusted'] = (master_df['average_teacher_salary'] * 100) / master_df['COLI']

# Remove COLI column
master_df = master_df.drop(columns=['COLI'])


### Suspensions
**Important Columns:**
- `TOT_DAYSMISSED_M` -> `suspensions_male` - School days missed due to out-of-school suspension: Calculated Male Total
- `TOT_DAYSMISSED_F` -> `suspensions_female` - School days missed due to out-of-school suspension: Calculated Female Total

**Exclusions:**
- n/a

In [28]:
path = os.path.join(CRDC_SCH_DATA_PATH, 'Suspensions.csv')
suspensions = pd.read_csv(path, encoding='windows-1252')

suspensions.head()

# Merge important columns into the master df
IMPORTANT_COLUMNS = ['TOT_DAYSMISSED_M', 'TOT_DAYSMISSED_F']
master_df = merge_with_combo_key(master_df, path, IMPORTANT_COLUMNS)

master_df = master_df.rename(columns={'TOT_DAYSMISSED_M': 'suspensions_male'})
master_df = master_df.rename(columns={'TOT_DAYSMISSED_F': 'suspensions_female'})

/var/folders/50/ps11njl93r52n622qd1nd_h80000gn/T/ipykernel_3888/2498991575.py:2: DtypeWarning: Columns (2,6) have mixed types. Specify dtype option on import or set low_memory=False.
  suspensions = pd.read_csv(path, encoding='windows-1252')


### Harassment and Bullying
**Important Columns:**
- `SCH_HBALLEGATIONS_SEX` -> `harassment_sex` -	Allegations of harassment or bullying on the basis of sex
- `SCH_HBALLEGATIONS_RAC` -> `harassment_race` - Allegations of harassment or bullying on the basis of race,color,or national origin
- `SCH_HBALLEGATIONS_DIS` -> `harassment_disability` - Allegations of harassment or bullying on the basis of disability
- `SCH_HBALLEGATIONS_ORI` -> `harassment_orientation` - Allegations of harassment or bullying  on the basis of sexual orientation
- `SCH_HBALLEGATIONS_REL` -> `harassment_religion` - Allegations of harassment or bullying on the basis of religion

**Exclusions:**
- n/a

In [29]:
path = os.path.join(CRDC_SCH_DATA_PATH, 'Harassment and Bullying.csv')
bullying = pd.read_csv(path, encoding='windows-1252')

#print(enrollment.columns.values)
bullying.head()

# Merge important columns into the master df
IMPORTANT_COLUMNS = [
    'SCH_HBALLEGATIONS_SEX',
    'SCH_HBALLEGATIONS_RAC',
    'SCH_HBALLEGATIONS_DIS',
    'SCH_HBALLEGATIONS_ORI',
    'SCH_HBALLEGATIONS_REL'
]
master_df = merge_with_combo_key(master_df, path, IMPORTANT_COLUMNS)

master_df = master_df.rename(columns={'SCH_HBALLEGATIONS_SEX': 'harassment_sex'})
master_df = master_df.rename(columns={'SCH_HBALLEGATIONS_RAC': 'harassment_race'})
master_df = master_df.rename(columns={'SCH_HBALLEGATIONS_DIS': 'harassment_disability'})
master_df = master_df.rename(columns={'SCH_HBALLEGATIONS_ORI': 'harassment_orientation'})
master_df = master_df.rename(columns={'SCH_HBALLEGATIONS_REL': 'harassment_religion'})

/var/folders/50/ps11njl93r52n622qd1nd_h80000gn/T/ipykernel_3888/351467712.py:2: DtypeWarning: Columns (2,6) have mixed types. Specify dtype option on import or set low_memory=False.
  bullying = pd.read_csv(path, encoding='windows-1252')


### Chronic Absences
Chronic absenteeism is defined as being absent 15 or more school days during the school year. A student is absent if he or she is not physically on school grounds and is not participating in instruction or instruction-related activities at an approved off-grounds location for the school day. Chronically absent students include students who are absent for any reason (e.g., illness, suspension, the need to care for a family member), regardless of whether absences are excused or unexcused.

**Important Columns:**
- `TOTAL_STUDENTS_REPORTED_M` -> `chronic_absences_male` - Number of Male Total enrolled students for all race-ethnicity categories
- `TOTAL_STUDENTS_REPORTED_F` -> `chronic_absences_female` - Number of Female Total enrolled students for all race-ethnicity categories

**Exclusions:**
- Not all schools are present in this ed facts data (All of Vermont is excluded)

In [30]:
path = os.path.join(EDFACTS_SCH_DATA_PATH, 'ID 814 SCH - Chronic Absenteeism.csv')
IMPORTANT_COLUMNS = ['TOTAL_STUDENTS_REPORTED_M', 'TOTAL_STUDENTS_REPORTED_F']
suspensions = pd.read_csv(path, encoding='windows-1252', dtype={'NCESSCH': str}, usecols=IMPORTANT_COLUMNS + ['NCESSCH'])
suspensions = suspensions.rename(columns={'NCESSCH': 'COMBOKEY'})

#print(enrollment.columns.values)
suspensions.head()


# Merge important columns into the master df
master_df = master_df.merge(suspensions, on='COMBOKEY', how='inner')

master_df = master_df.rename(columns={'TOTAL_STUDENTS_REPORTED_M': 'chronic_absences_male'})
master_df = master_df.rename(columns={'TOTAL_STUDENTS_REPORTED_F': 'chronic_absences_female'})

### Standardized State Assessment
school-level achievement results for state assessments in mathematics and reading or language arts. The math and reading score midpoints are combined into a single score with PCA. This feature is used for prediction in the decision tree regression.

**Important Columns:**
- `combined_proficiency_score_2018`

**Exclusions:**
- Schools that have NaN in the column are removed from the data. (All of West Virginia)

In [31]:
# read in the dataset, ncessch must be read as string to preserve leading zeros
assessment_df = pd.read_csv('datasets/schools_edfacts_assessments_2018.csv', dtype={'ncessch': str})
# take only rows where filters are set to 99 (totals)
filter_cols = ['grade_edfacts', 'race', 'sex', 'lep', 'homeless', 'migrant', 'disability', 'econ_disadvantaged', 'foster_care', 'military_connected']
assessment_df = assessment_df[(assessment_df[filter_cols] == 99).all(axis=1)]
# use only combokey, pct midpoint, and math midpoint
assessment_df = assessment_df[['ncessch', 'read_test_pct_prof_midpt', 'math_test_pct_prof_midpt']]

In [32]:
# 1. Select the reading and math midpoints (replace column names if needed)
X = assessment_df[['read_test_pct_prof_midpt', 'math_test_pct_prof_midpt']]

# 2. Handle missing values and standardize data (optional)
# Replace special values with NaN and then drop/replace NaN based on needs
X = X.replace([-1, -2, -3], float('nan')).dropna()

# Standardize if desired
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# 3. Apply PCA
pca = PCA(n_components=1)  # Only need one component for combined proficiency score
combined_proficiency_score = pca.fit_transform(X_scaled)

# 4. Add the result back to the DataFrame for use as a feature or target variable
assessment_df['combined_proficiency_score_2018'] = pd.Series(combined_proficiency_score.flatten(), index=X.index)

# Filter and rename the columns
combined_df = assessment_df[['ncessch', 'combined_proficiency_score_2018']]
combined_df = combined_df.rename(columns={'ncessch': 'COMBOKEY'})

# Drop nan rows and merge with master_df
combined_df = combined_df.dropna(subset=['combined_proficiency_score_2018'])
master_df = master_df.merge(combined_df, on='COMBOKEY', how='inner')

In [33]:
from sklearn.preprocessing import StandardScaler
import pandas as pd

# 1. Select the reading and math midpoints (replace column names if needed)
X = assessment_df['read_test_pct_prof_midpt']

# Filter and rename the columns
reading_df = assessment_df[['ncessch', 'read_test_pct_prof_midpt']]
reading_df = reading_df.rename(columns={'read_test_pct_prof_midpt': 'reading_proficiency_score_2018'})
reading_df = reading_df.rename(columns={'ncessch': 'COMBOKEY'})

# Drop NaN rows and merge with master_df
reading_df = reading_df.dropna(subset=['reading_proficiency_score_2018'])
master_df = master_df.merge(reading_df, on='COMBOKEY', how='inner')

# 5. Define classes based on quantiles
# Define the quantiles for the low, normal, and high categories
low_threshold = reading_df['reading_proficiency_score_2018'].quantile(0.33)
high_threshold = reading_df['reading_proficiency_score_2018'].quantile(0.67)

# Assign classes based on these thresholds
def assign_class(score):
    if score < low_threshold:
        return 'Low'
    elif score > high_threshold:
        return 'High'
    else:
        return 'Normal'

# Apply the function to create a new class column
reading_df['reading_proficiency_class'] = reading_df['reading_proficiency_score_2018'].apply(assign_class)

# Merge the class column back into master_df
master_df = master_df.merge(reading_df[['COMBOKEY', 'reading_proficiency_class']], on='COMBOKEY', how='inner')

# The master_df now contains the 'proficiency_class' column with discrete classes.
print(master_df[['COMBOKEY', 'reading_proficiency_score_2018', 'reading_proficiency_class']].head())

       COMBOKEY  reading_proficiency_score_2018 reading_proficiency_class
0  010000500870                            37.0                    Normal
1  010000500871                            31.0                       Low
2  010000500879                            39.0                    Normal
3  010000500889                            41.0                    Normal
4  010000600193                            47.0                    Normal


In [34]:
from sklearn.preprocessing import StandardScaler
import pandas as pd

# 1. Select the reading and math midpoints (replace column names if needed)
X = assessment_df['math_test_pct_prof_midpt']

# Filter and rename the columns
math_df = assessment_df[['ncessch', 'math_test_pct_prof_midpt']]
math_df = math_df.rename(columns={'math_test_pct_prof_midpt': 'math_proficiency_score_2018'})
math_df = math_df.rename(columns={'ncessch': 'COMBOKEY'})

# Drop NaN rows and merge with master_df
math_df = math_df.dropna(subset=['math_proficiency_score_2018'])
master_df = master_df.merge(math_df, on='COMBOKEY', how='inner')

# 5. Define classes based on quantiles
# Define the quantiles for the low, normal, and high categories
low_threshold = math_df['math_proficiency_score_2018'].quantile(0.33)
high_threshold = math_df['math_proficiency_score_2018'].quantile(0.67)

# Assign classes based on these thresholds
def assign_class(score):
    if score < low_threshold:
        return 'Low'
    elif score > high_threshold:
        return 'High'
    else:
        return 'Normal'

# Apply the function to create a new class column
math_df['math_proficiency_class'] = math_df['math_proficiency_score_2018'].apply(assign_class)

# Merge the class column back into master_df
master_df = master_df.merge(math_df[['COMBOKEY', 'math_proficiency_class']], on='COMBOKEY', how='inner')

# The master_df now contains the 'proficiency_class' column with discrete classes.
print(master_df[['COMBOKEY', 'math_proficiency_score_2018', 'math_proficiency_class']].head())

       COMBOKEY  math_proficiency_score_2018 math_proficiency_class
0  010000500870                         47.0                 Normal
1  010000500871                         39.0                 Normal
2  010000500879                         41.0                 Normal
3  010000500889                         51.0                 Normal
4  010000600193                         50.0                 Normal


In [35]:
# Write the master df to a csv
master_df.to_csv('master_df.csv', index=False)